# Part 9: Deep Learning with Keras

**Course:** 2026 KMITL Data Analytics

This lesson turns the calculations from Part 8 into a complete Keras workflow: prepare images, build a dense network, compile, fit, inspect validation history, evaluate, predict, save and reload. We use small Fashion-MNIST subsets so a CPU can run the lesson. The first dataset load requires internet access.


## Learning objectives

1. Prepare image values and independent training, validation and test sets.
2. Build a Sequential model with an explicit input shape and Dense layers.
3. Match output activation, target format, loss, optimizer and metrics.
4. Read training and validation curves.
5. Use early stopping, model checkpoints and learning-rate reduction.
6. Evaluate a selected model, make predictions, and verify saving and loading.


## Required imports

We use TensorFlow's Keras API. Google Colab normally includes TensorFlow; locally, install the updated project requirements in a supported Python environment. A fixed seed improves repeatability, but device and library differences can still change exact scores. Model files go into a new temporary directory on each run, so rerunning will not overwrite earlier work.


In [ ]:
from pathlib import Path
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)
model_folder = Path(tempfile.mkdtemp(prefix='part09_keras_'))
print('TensorFlow:', tf.__version__)
print('Model files will be saved in:', model_folder)


## Dataset introduction

**Fashion-MNIST** contains 28 × 28 grayscale clothing images in ten classes. Keras downloads about 30 MB of compressed files on the first call and caches them in its dataset directory, normally `~/.keras/datasets/`. It provides 60,000 training images and 10,000 test images. No manual file upload is required.

We take 6,000 training images and 1,000 validation images from the original training split, with no overlap. We select 1,000 examples from the separate test split for one final evaluation. Stratification preserves class proportions. These subsets are for fast teaching, not a full benchmark.

Each pixel starts as an integer from 0 to 255. Dividing by 255 maps 0 to 0, 255 to 1, and 128 to about **0.502**. In general, `scaled_pixel = original_pixel / 255`. This fixed conversion learns no statistics from validation or test data. Labels remain integer class IDs from 0 to 9.


In [ ]:
(all_train_images, all_train_labels), (all_test_images, all_test_labels) = keras.datasets.fashion_mnist.load_data()
train_indices, validation_indices = train_test_split(
    np.arange(len(all_train_images)), train_size=6000, test_size=1000,
    stratify=all_train_labels, random_state=RANDOM_STATE)
_, test_indices = train_test_split(np.arange(len(all_test_images)), test_size=1000,
                                   stratify=all_test_labels, random_state=RANDOM_STATE)
train_images = all_train_images[train_indices].astype('float32') / 255.0
validation_images = all_train_images[validation_indices].astype('float32') / 255.0
test_images = all_test_images[test_indices].astype('float32') / 255.0
train_labels = all_train_labels[train_indices]
validation_labels = all_train_labels[validation_indices]
test_labels = all_test_labels[test_indices]
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
assert set(train_indices).isdisjoint(validation_indices)
assert train_images.shape == (6000, 28, 28)
assert 0 <= train_images.min() <= train_images.max() <= 1
print('Train:', train_images.shape, '| validation:', validation_images.shape, '| test:', test_images.shape)
print('Class counts in training:', np.bincount(train_labels, minlength=10))
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for class_id, ax in enumerate(axes.flat):
    index = np.flatnonzero(train_labels == class_id)[0]
    ax.imshow(train_images[index], cmap='gray', vmin=0, vmax=1)
    ax.set_title(class_names[class_id])
    ax.set_xlabel('Image column')
    ax.set_ylabel('Image row')
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('One training example from each class')
plt.tight_layout()
plt.show()


**Read the images:** each panel is one grayscale training example with its known class name. Some tops, shirts and coats look similar at this resolution. A network must learn useful combinations of pixel values rather than treating the class ID as a numeric amount. The class counts confirm balanced training subsets.


## 1. Build a Sequential dense network

A **Sequential** model applies layers one after another. `Input(shape=(28, 28))` describes one example and does not include the batch size. `Flatten` changes each image into 784 pixel features. A **Dense** layer connects every input feature to each output unit. The hidden Dense layer uses 64 ReLU units; the output has 10 softmax units, one per class.

For two inputs and three Dense units, there are 2 × 3 = 6 weights plus 3 biases, giving **9 parameters**. With m inputs and n units, `parameter_count = m * n + n`. Here the first Dense layer has 784 × 64 + 64 = **50,240** parameters; the output has 64 × 10 + 10 = **650**. Flatten has no learned parameters.

For example, input [1, 2], one weight column [0.5, 1] and bias −1 give 1 × 0.5 + 2 × 1 − 1 = **1.5** before activation. In general, for an input row x, weights W, biases b and activation a, a Dense layer computes `output = a(x @ W + b)`, the same weighted-sum idea from Part 8. Flattening is a useful baseline, but it does not preserve image neighborhoods explicitly; Part 10 will use convolutions.


In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax'),
], name='fashion_dense')
model.summary()
assert model.count_params() == 50240 + 650


## 2. Compile: choose the training rules

`compile` selects the **loss**, **optimizer** and reported **metrics**. It does not train the model. Our labels are integers, so we use **sparse categorical cross-entropy** with softmax probabilities. 'Sparse' here describes the integer target representation; it does not mean sparse pixels.

If the correct class receives probability 0.7, its loss is −ln(0.7) ≈ **0.357**. Giving it 0.1 gives about **2.303**. For correct class y and predicted probability p_y, `loss = -ln(p_y)`. Keras averages across examples. Because the output already uses softmax, `from_logits=False` is the matching setting. Raw unnormalized scores would need `from_logits=True` instead.

**Adam** is an optimizer that adapts updates using moving averages of gradients and their sizes. We start with learning rate 0.001. **Accuracy** counts whether the largest predicted probability belongs to the true class: 8 correct predictions out of 10 give 8/10 = **0.8**. In general, `accuracy = correct_predictions / total_predictions`. Accuracy is a metric, not the differentiable training loss.


In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])
example_loss = keras.losses.sparse_categorical_crossentropy(np.array([1]), np.array([[0.2, 0.7, 0.1]])).numpy()[0]
assert np.isclose(example_loss, -np.log(0.7))
print('Example three-class loss:', round(float(example_loss), 3))


## 3. Fit with independent validation data

`fit` performs forward passes, backpropagation and optimizer updates. A batch of 128 examples produces one update; 6,000 training examples give 47 batches per epoch, with a smaller final batch. Validation runs without updates after each epoch. We allow at most six epochs for a short lesson.

A `tf.data.Dataset` groups examples into batches. We shuffle only training data and keep validation and test order fixed. The small private thread pool avoids creating excessive background threads on limited computers; it does not change the learning rule.

Three **callbacks** act during training:
- **EarlyStopping:** monitor validation loss, stop after two epochs without improvement, and restore the best weights. It may not trigger in a short run.
- **ModelCheckpoint:** save the entire model whenever validation loss improves. The `.keras` file can be loaded later.
- **ReduceLROnPlateau:** halve the learning rate after one epoch without improvement, down to 0.00001. For example, 0.001 becomes 0.0005. It also may not trigger.

Validation guides these decisions; the test set does not. The best epoch can be earlier than the final epoch. We will explicitly load the best checkpoint before test evaluation.


In [ ]:
batch_size = 128
data_options = tf.data.Options()
data_options.threading.private_threadpool_size = 1
train_data = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_data = train_data.shuffle(len(train_images), seed=RANDOM_STATE).batch(batch_size).with_options(data_options)
validation_data = tf.data.Dataset.from_tensor_slices((validation_images, validation_labels)).batch(batch_size).with_options(data_options)
test_data = tf.data.Dataset.from_tensor_slices((test_images, test_labels)).batch(batch_size).with_options(data_options)
checkpoint_path = model_folder / 'best_dense.keras'
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(str(checkpoint_path), monitor='val_loss', save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-5),
]
history = model.fit(train_data, validation_data=validation_data, epochs=6, callbacks=callbacks, shuffle=False, verbose=2)
assert checkpoint_path.is_file()


In [ ]:
epochs = np.arange(1, len(history.history['loss']) + 1)
best_epoch = int(np.argmin(history.history['val_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, label in zip(axes, ['loss', 'accuracy'], ['Cross-entropy', 'Accuracy']):
    ax.plot(epochs, history.history[metric], 'o-', label='Training')
    ax.plot(epochs, history.history['val_' + metric], 'o-', label='Validation')
    ax.axvline(best_epoch, linestyle=':', color='gray', label='Best validation loss epoch')
    ax.set(title=f'Dense network: {label.lower()}', xlabel='Epoch', ylabel=label, xticks=epochs)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print('Best validation-loss epoch:', best_epoch)


**Read both curves:** loss measures probability quality, while accuracy counts the winning class. Training and validation may improve together at first. Falling training loss alongside rising validation loss is a warning of overfitting. The dotted line marks the model-selection epoch. Do not assume the short run must overfit or trigger every callback. Training metrics average changing weights within an epoch, while validation uses the weights at the epoch end.


## 4. Evaluate once, then inspect predictions

`evaluate` compares predictions with labels and returns loss and metrics without updating weights. `predict` returns model outputs; it does not need labels. For class probabilities [0.1, 0.7, 0.2], `argmax` chooses index **1**, the largest value. In general, the predicted class is the index of the largest class probability.

Load the checkpoint chosen by validation loss, then evaluate on the independent test subset. Do not choose a different model after seeing these test results. The displayed probabilities are model confidence, not a promise of correctness.


In [ ]:
best_model = keras.models.load_model(checkpoint_path)
test_results = best_model.evaluate(test_data, verbose=0, return_dict=True)
print('Held-out test results:', test_results)
prediction_data = tf.data.Dataset.from_tensor_slices(test_images[:10]).batch(10).with_options(data_options)
probabilities = best_model.predict(prediction_data, verbose=0)
predicted_labels = probabilities.argmax(axis=1)
assert probabilities.shape == (10, 10)
assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for index, ax in enumerate(axes.flat):
    actual = class_names[test_labels[index]]
    predicted_name = class_names[predicted_labels[index]]
    ax.imshow(test_images[index], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'True: {actual}\nPred: {predicted_name}', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('First ten test examples: true and predicted classes')
plt.tight_layout()
plt.show()


**Read the predictions:** each title states the true label and the predicted class, so agreement and disagreement do not depend on color. These ten images are an inspection sample, not the full test metric. Similar clothing shapes can confuse the model. A different runtime may produce somewhat different predictions, especially near uncertain boundaries.


## 5. Save and reload the selected model

A `.keras` file stores the architecture, weights and training configuration, including optimizer state when present. Saving only weights would require rebuilding the architecture separately. We save the selected best model, load it again, and compare outputs on the same input. They should agree within a small numerical tolerance.

The pixel scaling and class-name list are outside this model, so an application must preserve that preprocessing and class mapping too. Temporary files may disappear when the system cleans its temporary directory or a Colab runtime ends. Copy or download the printed model files if you want to keep them. Never load a model file from an untrusted source.


In [ ]:
saved_path = model_folder / 'selected_dense.keras'
best_model.save(saved_path)
reloaded_model = keras.models.load_model(saved_path)
reloaded_probabilities = reloaded_model(test_images[:10], training=False).numpy()
assert np.allclose(probabilities, reloaded_probabilities, atol=1e-6)
assert saved_path.is_file()
assert all(np.isfinite(history.history[key]).all() for key in ['loss', 'val_loss', 'accuracy', 'val_accuracy'])
assert np.isfinite(test_results['loss']) and 0 <= test_results['accuracy'] <= 1
print('Saved and reloaded:', saved_path)
print('Keras workflow checks passed.')


## Common mistakes

- Including batch size in the per-example Input shape, or forgetting the 255 scaling.
- Using integer labels with a loss that expects one-hot vectors.
- Combining softmax outputs with `from_logits=True`.
- Thinking `compile` trains, or that `predict` calculates accuracy.
- Monitoring test loss in callbacks instead of validation loss.
- Reporting the final epoch as the best without checking validation history.
- Saving a model but forgetting its preprocessing, label mapping or temporary-file lifetime.


## Summary

The Keras workflow separates model construction, training rules, fitting and evaluation. Validation history guides stopping, learning-rate changes and checkpoint selection. A separate test set measures the selected model once. Predictions and saved-model checks complete the workflow. This dense image classifier is a baseline; convolutional layers will make better use of image structure in Part 10.
